# Лабораторная работа № 3

In [1]:
# Установка необходимых библиотек
# Раскомментируйте и выполните при первом запуске

# !pip install sqlalchemy pymorphy2 scikit-learn matplotlib seaborn
# !pip install fastapi uvicorn httpx pydantic
# !pip install wordcloud

import warnings
warnings.filterwarnings('ignore')

import os, re, json, random, time
from datetime import date, timedelta
from collections import Counter
from typing import Optional, List

import numpy as np
import matplotlib
matplotlib.use('Agg')  # для серверного окружения
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import (
    create_engine, Column, Integer, String, Text, Date,
    Float, ForeignKey, func, Index, event
)
from sqlalchemy.orm import (
    declarative_base, relationship, sessionmaker, Session
)

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, accuracy_score
)

try:
    import pymorphy3 as pymorphy2
    morph = pymorphy2.MorphAnalyzer()
    PYMORPHY_OK = True
except ImportError:
    PYMORPHY_OK = False
    print('pymorphy2 недоступен: лемматизация будет упрощённой')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

print('Все библиотеки импортированы.')

Все библиотеки импортированы.


In [2]:
# ============================================================
# ORM-модели юридической базы данных
# ============================================================

Base = declarative_base()

class Court(Base):
    __tablename__ = 'courts'
    id     = Column(Integer, primary_key=True)
    name   = Column(String(255), nullable=False)
    region = Column(String(100))
    level  = Column(String(50))
    cases  = relationship('Case', back_populates='court')

class Case(Base):
    __tablename__ = 'cases'
    id          = Column(Integer, primary_key=True)
    number      = Column(String(50), nullable=False, unique=True)
    court_id    = Column(Integer, ForeignKey('courts.id'), nullable=False)
    date_filed  = Column(Date, nullable=False)
    category    = Column(String(100))
    status      = Column(String(30), default='завершено')
    court        = relationship('Court', back_populates='cases')
    documents    = relationship('Document', back_populates='case',
                                cascade='all, delete-orphan')
    participants = relationship('Participant', back_populates='case',
                                cascade='all, delete-orphan')

class Participant(Base):
    __tablename__ = 'participants'
    id      = Column(Integer, primary_key=True)
    case_id = Column(Integer, ForeignKey('cases.id'), nullable=False)
    name    = Column(String(255), nullable=False)
    role    = Column(String(50), nullable=False)
    case    = relationship('Case', back_populates='participants')

class Document(Base):
    __tablename__ = 'documents'
    id       = Column(Integer, primary_key=True)
    title    = Column(String(500), nullable=False)
    date     = Column(Date, nullable=False)
    text     = Column(Text, nullable=False)
    doc_type = Column(String(50), nullable=False)
    case_id  = Column(Integer, ForeignKey('cases.id'), nullable=False)
    case        = relationship('Case', back_populates='documents')
    entities    = relationship('Entity', back_populates='document',
                               cascade='all, delete-orphan')
    annotations = relationship('Annotation', back_populates='document',
                               cascade='all, delete-orphan')

class Entity(Base):
    __tablename__ = 'entities'
    id          = Column(Integer, primary_key=True)
    document_id = Column(Integer, ForeignKey('documents.id'), nullable=False)
    name        = Column(String(255), nullable=False)
    ner_type    = Column(String(50), nullable=False)
    confidence  = Column(Float, default=1.0)
    document    = relationship('Document', back_populates='entities')

class Annotation(Base):
    __tablename__ = 'annotations'
    id          = Column(Integer, primary_key=True)
    document_id = Column(Integer, ForeignKey('documents.id'), nullable=False)
    label       = Column(String(100), nullable=False)
    confidence  = Column(Float, default=1.0)
    model_name  = Column(String(100))
    document    = relationship('Document', back_populates='annotations')


# ============================================================
# Конфигурация корпусов
# ============================================================
COURTS_DATA = [
    ('Арбитражный суд г. Москвы', 'Москва', 'районный'),
    ('Арбитражный суд Московской области', 'Московская обл.', 'районный'),
    ('Девятый арбитражный апелляционный суд', 'Москва', 'апелляционный'),
    ('Арбитражный суд Санкт-Петербурга', 'Санкт-Петербург', 'районный'),
    ('Тринадцатый ААС', 'Санкт-Петербург', 'апелляционный'),
    ('Арбитражный суд Свердловской области', 'Свердловская обл.', 'районный'),
    ('Арбитражный суд Новосибирской области', 'Новосибирская обл.', 'районный'),
]

CATEGORIES = {
    'A': {'label': 'Гражданское судопроизводство',
          'classes': ['трудовые споры','семейные дела','жилищные споры',
                      'имущественные претензии','защита прав потребителей']},
    'B': {'label': 'Арбитражное судопроизводство',
          'classes': ['банкротство','налоговые споры','корпоративные конфликты',
                      'интеллектуальная собственность','договорные споры']},
    'C': {'label': 'Административное судопроизводство',
          'classes': ['оспаривание решений органов','земельные споры',
                      'лицензионные дела','миграционные дела','таможенные споры']},
    'D': {'label': 'Уголовное судопроизводство',
          'classes': ['мошенничество','кража','должностные преступления',
                      'экономические преступления','нарушение ПДД']},
    'E': {'label': 'Смешанный корпус',
          'classes': ['банкротство','трудовые споры','налоговые споры',
                      'корпоративные конфликты','жилищные споры']},
}

ORGS = ['ООО «Альфа»','ПАО «Бета-Банк»','АО «Гамма Групп»','ООО «Дельта Строй»',
        'ПАО «Эпсилон»','ООО «Зета Консалт»','АО «Тета Лоджистик»','ООО «Сигма»']
DOC_TYPES = ['решение','определение','постановление']


def _make_text(cls_name, rng):
    """Генерация синтетического текста судебного акта."""
    templates = [
        "Суд рассмотрел дело по категории {c}. В ходе судебного заседания "
        "были исследованы материалы дела и заслушаны показания сторон. "
        "Истец заявил требования о защите нарушенных прав в сфере {c}.",
        "Арбитражный суд установил следующее по делу о {c}. "
        "Заявитель обратился с исковым заявлением, обоснованным нормами "
        "действующего законодательства Российской Федерации.",
        "По результатам рассмотрения спора в области {c} суд пришёл "
        "к выводу о необходимости удовлетворения заявленных требований. "
        "Нормативной основой послужили положения Гражданского кодекса.",
    ]
    base = rng.choice(templates).format(c=cls_name)
    extra = ' '.join(rng.choices(
        [cls_name, 'суд', 'решение', 'истец', 'ответчик', 'закон',
         'право', 'обязательство', 'ущерб', 'компенсация', 'иск',
         'апелляция', 'кодекс', 'статья', 'доказательство'], k=rng.randint(20,50)))
    return f'{base} {extra}'


def generate_corpus_db(corpus_type, n_per_class=60, seed=42):
    """Генерирует корпус и наполняет SQLite БД через ORM."""
    rng = random.Random(seed)
    db_path = 'legal_ips_variant.db'
    if os.path.exists(db_path):
        os.remove(db_path)

    engine = create_engine(f'sqlite:///{db_path}', echo=False)
    Base.metadata.create_all(engine)
    Sess = sessionmaker(bind=engine)
    session = Sess()

    court_objs = []
    for name, region, level in COURTS_DATA:
        c = Court(name=name, region=region, level=level)
        session.add(c); court_objs.append(c)
    session.flush()

    cat_info = CATEGORIES[corpus_type]
    all_docs = []

    for cls_name in cat_info['classes']:
        for _ in range(n_per_class):
            court = rng.choice(court_objs)
            year = rng.choice([2021,2022,2023,2024])
            case_num = f'A{rng.randint(10,99)}-{rng.randint(10000,99999)}/{year}'

            case_obj = Case(number=case_num, court_id=court.id,
                           date_filed=date(year, rng.randint(1,12), rng.randint(1,28)),
                           category=cls_name,
                           status=rng.choice(['завершено','на обжаловании']))
            session.add(case_obj); session.flush()

            p1 = Participant(case_id=case_obj.id, name=rng.choice(ORGS), role='истец')
            p2 = Participant(case_id=case_obj.id, name=rng.choice(ORGS), role='ответчик')
            session.add_all([p1, p2])

            full_text = _make_text(cls_name, rng)
            doc_obj = Document(
                title=f'{rng.choice(DOC_TYPES).capitalize()} по делу {case_num} ({cls_name})',
                date=date(year, rng.randint(1,12), rng.randint(1,28)),
                text=full_text, doc_type=rng.choice(DOC_TYPES), case_id=case_obj.id)
            session.add(doc_obj); session.flush()
            all_docs.append({'doc_id': doc_obj.id, 'text': full_text,
                            'label': cls_name, 'case_number': case_num})

    session.commit()
    print(f"БД создана: {db_path}")
    print(f"  Корпус: {cat_info['label']}, документов: {len(all_docs)}, "
          f"классов: {len(cat_info['classes'])}")
    return engine, session, all_docs


# Функции предобработки
_STOP = set('и в на по с к от до из за о об для при что как не но а или это то же ещё '
            'бы ли уже да нет он она оно они мы вы так этот тот все весь был быть будет '
            'свой его её их'.split())

def preprocess_text(text):
    text = re.sub(r'[^а-яёА-ЯЁa-zA-Z\s]', ' ', text.lower())
    tokens = [t for t in text.split() if len(t) > 2 and t not in _STOP]
    if PYMORPHY_OK:
        tokens = [morph.parse(t)[0].normal_form for t in tokens]
    return ' '.join(tokens)

def simple_ner(text):
    entities = []
    for m in re.finditer(r'(?:ООО|ПАО|АО|ЗАО)\s*[«"][^»"]+[»"]', text):
        entities.append({'name': m.group(), 'type': 'ORG'})
    for m in re.finditer(r'\d[\d\s]+(?:рубл|руб\.)', text):
        entities.append({'name': m.group().strip(), 'type': 'MONEY'})
    for m in re.finditer(r'стать[яиеюёй]+\s+[\d.]+', text):
        entities.append({'name': m.group().strip(), 'type': 'LAW_REF'})
    return entities

print("ORM-модели, генератор и функции предобработки определены.")
print(f"Доступные корпуса: {list(CATEGORIES.keys())}")


ORM-модели, генератор и функции предобработки определены.
Доступные корпуса: ['A', 'B', 'C', 'D', 'E']


### Задание 1. Проектирование и наполнение базы данных (для всех вариантов)

In [3]:
# ============================================================
# Задание 1: Проектирование и наполнение БД
# ============================================================
# ВНИМАНИЕ: замените параметры согласно варианту!

CORPUS_TYPE = 'C'   # <-- тип корпуса из таблицы вариантов
VARIANT_NUMBER = 19  # <-- номер варианта

# -- 1.1 Создание БД и генерация корпуса --
engine, session, docs_data = generate_corpus_db(
    CORPUS_TYPE, n_per_class=60, seed=RANDOM_STATE)

# -- 1.2 Статистика по таблицам --
print("\n=== Статистика базы данных ===")
for model, name in [(Court,'courts'), (Case,'cases'),
                     (Participant,'participants'), (Document,'documents')]:
    count = session.query(func.count(model.id)).scalar()
    print(f"  {name:15s}: {count} записей")

БД создана: legal_ips_variant.db
  Корпус: Административное судопроизводство, документов: 300, классов: 5

=== Статистика базы данных ===
  courts         : 7 записей
  cases          : 300 записей
  participants   : 600 записей
  documents      : 300 записей


In [4]:
# -- 1.3 Распределение по классам --
print("\n=== Распределение по категориям дел ===")
cat_stats = (session.query(Case.category, func.count(Document.id))
             .join(Document).group_by(Case.category).all())
for cat, cnt in cat_stats:
    print(f"  {cat:30s}: {cnt}")


=== Распределение по категориям дел ===
  земельные споры               : 60
  лицензионные дела             : 60
  миграционные дела             : 60
  оспаривание решений органов   : 60
  таможенные споры              : 60


In [5]:
# -- 1.4 Визуализация --
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cats, cnts = zip(*cat_stats) if cat_stats else ([], [])
axes[0].barh(list(cats), list(cnts), color='steelblue', edgecolor='white')
axes[0].set_title('Распределение по категориям')
axes[0].set_xlabel('Число документов')

court_stats = (session.query(Court.name, func.count(Document.id))
               .join(Case, Court.id == Case.court_id)
               .join(Document).group_by(Court.name).all())
if court_stats:
    cn, cc = zip(*court_stats)
    axes[1].barh(list(cn), list(cc), color='teal', edgecolor='white')
axes[1].set_title('Распределение по судам')
plt.suptitle(f'Анализ БД: {CATEGORIES[CORPUS_TYPE]["label"]}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('task1_db_stats.png', dpi=120, bbox_inches='tight')
plt.show()

In [6]:

# -- 1.5 Пример ORM-запроса --
print("\n=== Документы первого суда за 2024 год ===")
sample = (session.query(Document).join(Case).join(Court)
          .filter(Court.id == 1, Case.date_filed >= date(2024,1,1))
          .limit(5).all())
for d in sample:
    print(f"  [{d.doc_type}] {d.title[:60]}...")


=== Документы первого суда за 2024 год ===
  [решение] Постановление по делу A31-78712/2024 (оспаривание решений ор...
  [решение] Постановление по делу A94-27082/2024 (земельные споры)...
  [определение] Определение по делу A57-91352/2024 (земельные споры)...
  [постановление] Определение по делу A55-27659/2024 (земельные споры)...
  [постановление] Решение по делу A74-30195/2024 (земельные споры)...


In [7]:
"""База данных юридического корпуса построена по реляционной схеме «снежинка» и включает
шесть таблиц. Центральная сущность — Case (дело), которая связана с Court (суд) через
внешний ключ court_id: одному суду соответствует множество дел (отношение один-ко-многим).
К каждому делу через case_id привязаны две дочерние таблицы с каскадным удалением:
Participant (участники процесса — истец и ответчик) и Document (судебные акты).
Это обеспечивает ссылочную целостность: при удалении дела все связанные документы
и участники удаляются автоматически. Каждый документ, в свою очередь, является
родителем для двух таблиц: Entity (именованные сущности, извлечённые NER-модулем —
организации, суммы, ссылки на статьи) и Annotation (метки классификатора с оценкой
уверенности и именем модели). Оба отношения также каскадные.

Ограничения целостности: Case.number объявлен UNIQUE — дублирование номеров дел
исключено на уровне СУБД. Поля Court.name, Case.number, Document.title, Document.text,
Document.doc_type, Participant.name и Participant.role объявлены NOT NULL, что
гарантирует отсутствие неполных записей. Поле Document.doc_type де-факто принимает
только три значения (решение, определение, постановление); в production-системе это
следует закрепить CHECK-ограничением или вынести в отдельный справочник. Поле
Entity.confidence и Annotation.confidence имеют DEFAULT 1.0, что позволяет добавлять
записи без явного указания уверенности при ручной разметке. Описанная схема
обеспечивает нормальную форму 3NF, минимизирует избыточность и позволяет строить
агрегатные запросы по судам, категориям дел и типам актов средствами SQLAlchemy ORM."""

'База данных юридического корпуса построена по реляционной схеме «снежинка» и включает\nшесть таблиц. Центральная сущность — Case (дело), которая связана с Court (суд) через\nвнешний ключ court_id: одному суду соответствует множество дел (отношение один-ко-многим).\nК каждому делу через case_id привязаны две дочерние таблицы с каскадным удалением:\nParticipant (участники процесса — истец и ответчик) и Document (судебные акты).\nЭто обеспечивает ссылочную целостность: при удалении дела все связанные документы\nи участники удаляются автоматически. Каждый документ, в свою очередь, является\nродителем для двух таблиц: Entity (именованные сущности, извлечённые NER-модулем —\nорганизации, суммы, ссылки на статьи) и Annotation (метки классификатора с оценкой\nуверенности и именем модели). Оба отношения также каскадные.\n\nОграничения целостности: Case.number объявлен UNIQUE — дублирование номеров дел\nисключено на уровне СУБД. Поля Court.name, Case.number, Document.title, Document.text,\nDocu

### Задание 2. NLP-конвейер: предобработка, NER и классификация (для всех вариантов)

In [8]:
# ============================================================
# Задание 2: NLP-конвейер
# ============================================================
import pandas as pd

# -- 2.1 Предобработка текстов --
df = pd.DataFrame(docs_data)
df['processed'] = df['text'].apply(preprocess_text)
print(f"Документов: {len(df)}, средняя длина: "
      f"{df['processed'].apply(lambda x: len(x.split())).mean():.0f} токенов")

Документов: 300, средняя длина: 58 токенов


In [10]:
# -- 2.2 NER: извлечение именованных сущностей --
print("\n=== Извлечение именованных сущностей ===")
total_ents = 0
for _, row in df.iterrows():
    ents = simple_ner(row['text'])
    for e in ents:
        ent_obj = Entity(document_id=row['doc_id'],
                         name=e['name'], ner_type=e['type'], confidence=0.85)
        session.add(ent_obj)
    total_ents += len(ents)
session.commit()
print(f"Сохранено сущностей: {total_ents}")

# Статистика NER по типам
ner_stats = (session.query(Entity.ner_type, func.count(Entity.id))
             .group_by(Entity.ner_type).all())
for t, c in ner_stats:
    print(f"  {t}: {c}")


=== Извлечение именованных сущностей ===
Сохранено сущностей: 0


In [13]:
# -- 2.3 Классификация --
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['processed'], df['label'], test_size=0.2,
    random_state=RANDOM_STATE, stratify=df['label'])

# ВНИМАНИЕ: настройте векторизатор согласно варианту!
# Варианты TF-IDF unigrams:
# vectorizer = TfidfVectorizer(ngram_range=(1,1), max_features=3000,
#                               sublinear_tf=True, min_df=2)
# Для BoW: vectorizer = CountVectorizer(max_features=2000, min_df=2)
# Для bigrams: TfidfVectorizer(ngram_range=(2,2), ...)
# Для char: CountVectorizer(analyzer='char_wb', ngram_range=(3,5), ...)
vectorizer = CountVectorizer(analyzer='char_wb', max_features=2000, min_df=2, ngram_range=(3, 5))

X_train = vectorizer.fit_transform(X_train_raw)
X_test  = vectorizer.transform(X_test_raw)
print(f"\nПризнаковое пространство: {X_train.shape}")
print(f"\nПризнаковое пространство: {X_test.shape}")

# ВНИМАНИЕ: выберите классификатор согласно варианту!
# clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
# Для MultinomialNB: clf = MultinomialNB()
# Для RandomForest:  clf = RandomForestClassifier(100, random_state=RANDOM_STATE)
# Для LinearSVC:     clf = LinearSVC(max_iter=2000, random_state=RANDOM_STATE)
# Для GradientBoosting: clf = GradientBoostingClassifier(random_state=RANDOM_STATE)
clf = LinearSVC(max_iter=2000, random_state=RANDOM_STATE, C=1.0)

# Кросс-валидация
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(clf, X_train, y_train, cv=cv, scoring='f1_weighted')
print(f"\nCV F1-weighted: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

# Обучение и предсказание
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

# Матрица ошибок
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=clf.classes_, yticklabels=clf.classes_, ax=ax)
ax.set_title('Матрица ошибок классификатора')
ax.set_xlabel('Предсказано'); ax.set_ylabel('Истинно')
plt.tight_layout()
plt.savefig('task2_confusion.png', dpi=120, bbox_inches='tight')
plt.show()


Признаковое пространство: (240, 1085)

Признаковое пространство: (60, 1085)

CV F1-weighted: 0.979 (+/- 0.023)

=== Classification Report ===
                             precision    recall  f1-score   support

            земельные споры       1.00      1.00      1.00        12
          лицензионные дела       1.00      1.00      1.00        12
          миграционные дела       1.00      1.00      1.00        12
оспаривание решений органов       1.00      1.00      1.00        12
           таможенные споры       1.00      1.00      1.00        12

                   accuracy                           1.00        60
                  macro avg       1.00      1.00      1.00        60
               weighted avg       1.00      1.00      1.00        60



In [14]:
# -- 2.4 Сохранение аннотаций в БД --
for doc_id, pred_label in zip(df.loc[X_test_raw.index, 'doc_id'], y_pred):
    ann = Annotation(document_id=int(doc_id), label=pred_label,
                     confidence=float(f1_score(y_test, y_pred, average='weighted')),
                     model_name=type(clf).__name__)
    session.add(ann)
session.commit()
print(f"\nАннотации сохранены в БД: {len(y_pred)} записей")


Аннотации сохранены в БД: 60 записей


### Задание 3. Поисковая подсистема (тип поиска определяется вариантом)

In [18]:
def search_fulltext(session, query_text, doc_type_filter=None, limit=10):
    """Полнотекстовый поиск по содержанию документа (LIKE-имитация для SQLite)."""
    q = session.query(Document).join(Case)
    terms = query_text.lower().split()
    for term in terms:
        q = q.filter(Document.text.ilike(f'%{term}%'))
    if doc_type_filter:
        q = q.filter(Document.doc_type == doc_type_filter)
    results = q.limit(limit).all()
    # Формирование сниппетов
    output = []
    for doc in results:
        idx = doc.text.lower().find(terms[0])
        start = max(0, idx - 50)
        end = min(len(doc.text), idx + 80)
        snippet = '...' + doc.text[start:end] + '...'
        output.append({'id': doc.id, 'title': doc.title,
                       'date': str(doc.date), 'type': doc.doc_type,
                       'snippet': snippet})
    return output

print("=== Тестовый запрос 1: полнотекстовый поиск по тексту ===")
results1 = search_fulltext(session, query_text='миграционные', limit=3)
for r in results1:
    print(f"  [{r['type']}] {r['title'][:60]}...")
    print(f"    Сниппет: {r['snippet'][:80]}...")

print("=== Тестовый запрос 2: полнотекстовый поиск по тексту и типу ===")
results2 = search_fulltext(session, "таможенные споры", doc_type_filter='решение', limit=3)
for r in results2:
    print(f"  [{r['type']}] {r['title'][:60]}...")
    print(f"    Сниппет: {r['snippet'][:80]}...")
    
print("=== Тестовый запрос 3: полнотекстовый поиск по тексту и типу ===")
results3 = search_fulltext(session, CATEGORIES[CORPUS_TYPE]['classes'][2], doc_type_filter='определение', limit=3)
for r in results3:
    print(f"  [{r['type']}] {r['title'][:60]}...")
    print(f"    Сниппет: {r['snippet'][:80]}...")

=== Тестовый запрос 1: полнотекстовый поиск по тексту ===
  [решение] Решение по делу A96-79688/2023 (миграционные дела)...
    Сниппет: ...Арбитражный суд установил следующее по делу о миграционные дела. Заявитель об...
  [решение] Определение по делу A50-21814/2024 (миграционные дела)...
    Сниппет: ...Суд рассмотрел дело по категории миграционные дела. В ходе судебного заседани...
  [определение] Определение по делу A53-99006/2021 (миграционные дела)...
    Сниппет: ...Арбитражный суд установил следующее по делу о миграционные дела. Заявитель об...
=== Тестовый запрос 2: полнотекстовый поиск по тексту и типу ===
  [решение] Решение по делу A35-57652/2022 (таможенные споры)...
    Сниппет: ...По результатам рассмотрения спора в области таможенные споры суд пришёл к выв...
  [решение] Постановление по делу A41-86765/2022 (таможенные споры)...
    Сниппет: ...Арбитражный суд установил следующее по делу о таможенные споры. Заявитель обр...
  [решение] Постановление по делу A66-42142/20

### Задание 4. REST API для доступа к системе (для всех вариантов)

In [20]:
# ============================================================
# Задание 4: REST API (FastAPI)
# ============================================================
# Примечание: для полноценного запуска требуется pip install fastapi httpx

try:
    from pydantic import BaseModel, Field
    from typing import Optional, List as TList
    PYDANTIC_OK = True
except ImportError:
    PYDANTIC_OK = False
    print("pydantic недоступен: определите схемы вручную")

if PYDANTIC_OK:
    class DocumentOut(BaseModel):
        id: int
        title: str
        date: str
        doc_type: str
        snippet: Optional[str] = None

        class Config:
            from_attributes = True

    class SearchRequest(BaseModel):
        query: str = Field(..., min_length=2, max_length=500)
        doc_type: Optional[str] = None
        limit: int = Field(default=10, ge=1, le=100)

    class SearchResponse(BaseModel):
        total: int
        results: TList[DocumentOut]

# -- Определение API (структурный пример) --
api_code = '''
from fastapi import FastAPI, Depends, Query, HTTPException

app = FastAPI(title="Юридическая ИПС", version="1.0")

@app.get("/api/documents", response_model=list[DocumentOut])
def list_documents(skip: int = 0, limit: int = 20):
    docs = session.query(Document).offset(skip).limit(limit).all()
    return [DocumentOut(id=d.id, title=d.title, date=str(d.date),
                        doc_type=d.doc_type) for d in docs]

@app.get("/api/documents/{doc_id}", response_model=DocumentOut)
def get_document(doc_id: int):
    doc = session.query(Document).filter(Document.id == doc_id).first()
    if not doc:
        raise HTTPException(404, "Документ не найден")
    return DocumentOut(id=doc.id, title=doc.title, date=str(doc.date),
                       doc_type=doc.doc_type)

@app.post("/api/search", response_model=SearchResponse)
def search(req: SearchRequest):
    results = search_fulltext(session, req.query,
                              doc_type_filter=req.doc_type, limit=req.limit)
    return SearchResponse(total=len(results),
                          results=[DocumentOut(**r) for r in results])
'''

print("Структура REST API юридической ИПС")
print("=" * 60)
print(api_code)


Структура REST API юридической ИПС

from fastapi import FastAPI, Depends, Query, HTTPException

app = FastAPI(title="Юридическая ИПС", version="1.0")

@app.get("/api/documents", response_model=list[DocumentOut])
def list_documents(skip: int = 0, limit: int = 20):
    docs = session.query(Document).offset(skip).limit(limit).all()
    return [DocumentOut(id=d.id, title=d.title, date=str(d.date),
                        doc_type=d.doc_type) for d in docs]

@app.get("/api/documents/{doc_id}", response_model=DocumentOut)
def get_document(doc_id: int):
    doc = session.query(Document).filter(Document.id == doc_id).first()
    if not doc:
        raise HTTPException(404, "Документ не найден")
    return DocumentOut(id=doc.id, title=doc.title, date=str(doc.date),
                       doc_type=doc.doc_type)

@app.post("/api/search", response_model=SearchResponse)
def search(req: SearchRequest):
    results = search_fulltext(session, req.query,
                              doc_type_filter=re

In [21]:
# -- Тестирование API программно --
print("\n=== Тестирование эндпойнтов (имитация) ===")

# Имитация GET /api/documents
docs_list = session.query(Document).limit(3).all()
print(f"GET /api/documents => {len(docs_list)} документов")
for d in docs_list:
    print(f"  id={d.id}, type={d.doc_type}, date={d.date}")

# Имитация GET /api/documents/{id}
doc = session.query(Document).first()
print(f"\nGET /api/documents/{doc.id} => '{doc.title[:50]}...'")

# Имитация POST /api/search
query_term = CATEGORIES[CORPUS_TYPE]['classes'][0]
results = search_fulltext(session, query_term, limit=3)
print(f"\nPOST /api/search (query='{query_term}') => {len(results)} результатов")
for r in results:
    print(f"  [{r['type']}] {r['title'][:50]}...")

# TODO: При наличии FastAPI запустите реальный сервер и TestClient


=== Тестирование эндпойнтов (имитация) ===
GET /api/documents => 3 документов
  id=1, type=определение, date=2021-04-25
  id=2, type=решение, date=2022-05-05
  id=3, type=определение, date=2023-01-20

GET /api/documents/1 => 'Постановление по делу A13-46048/2021 (оспаривание ...'

POST /api/search (query='оспаривание решений органов') => 3 результатов
  [определение] Постановление по делу A13-46048/2021 (оспаривание ...
  [решение] Решение по делу A22-59823/2022 (оспаривание решени...
  [определение] Постановление по делу A84-66155/2023 (оспаривание ...


### Задание 5. Специализированное задание (определяется вариантом)

In [22]:
# ── 5.3. Экспорт в JSON (варианты 3, 11, 19, 26, 35) ────────
export = []
for case in session.query(Case).all():
    case_data = {'number': case.number, 'category': case.category,
                 'court': case.court.name, 'documents': [], 'participants': []}
    for p in case.participants:
        case_data['participants'].append({'name': p.name, 'role': p.role})
    for d in case.documents:
        doc_data = {'title': d.title, 'date': str(d.date), 'type': d.doc_type,
                    'entities': [{'name':e.name,'type':e.ner_type} for e in d.entities],
                    'annotations': [{'label':a.label} for a in d.annotations]}
        case_data['documents'].append(doc_data)
    export.append(case_data)
with open('export.json','w',encoding='utf-8') as f:
    json.dump(export, f, ensure_ascii=False, indent=2)
print(f"Экспортировано: {len(export)} дел, размер: {os.path.getsize('export.json')/1024:.1f} КБ")

Экспортировано: 300 дел, размер: 208.2 КБ
